## What is new in models4 (vs models3)

- **Batched graph construction** via `build_batch` — graphs built slice by slice, so training starts after the first batch rather than waiting for the full dataset.
- **Evidence retry** via `retry_failed_evidence` — three fallback strategies (truncated headline → NER keywords → lead sentence) recover articles that initially return zero search results.
- **Warm-start fine-tuning** via `finetune_gat` — each subsequent batch fine-tunes from the previous model's weights at a lower LR.
- **Compounding credibility DB** — the source database is updated after every batch, enriching evidence scoring for later batches.
- **Visualised learning curve** — per-batch accuracy / F1 / AUC on a fixed global validation pool.

---

# models4.ipynb — Batched Build & Incremental Training

## Overview

This notebook builds a **Graph Attention Network (GAT)** for fake-news detection on the
[ISOT dataset](https://www.kaggle.com/datasets/emineyetm/fake-news-detection-datasets).
It is the third iteration in this series; see `models2.ipynb` for the previous version.

Each news article is converted into a **Structured Argumentation Graph**: a typed graph
where nodes represent sentences (labelled by rhetorical role) and edges encode semantic
similarity and logical relations (entailment / contradiction / neutral) both within the
article and across externally retrieved articles covering the same event.

---

## Core redesign from models2

### What was wrong
`models2` searched using entity queries extracted from the article body. This produced
irrelevant results (emojipedia, dndbeyond, tacomaworld) and left **262 / 500 graphs with
zero evidence nodes** — graphs that all look structurally identical and give the model
nothing to learn from.

### What changes

**1. Headline-based search** *(O(1) per article, not O(n_claims))*  
Headlines are written to be precise and findable. Searching the headline directly
retrieves articles covering the *same event* from different publishers — exactly the
cross-source comparison we want.

**2. Sentence role classification** *(zero-shot NLI)*  
Each sentence is labelled as one of: `claim`, `evidence`, `analysis`, `background`.
Misinformation manipulates the *analysis* layer while keeping evidence plausible — so
this distinction is the key signal that was missing.

**3. Cross-source analysis comparison**  
The target article's analysis sentences are compared via NLI to analysis sentences from
retrieved articles covering the same topic. Analysis entailed by multiple independent
sources is credible; analysis that contradicts them is a fake signal.

**4. Typed graph structure**  
Nodes carry role labels. Edges connect: intra-article (sentence↔sentence) and
cross-source (target analysis ↔ retrieved analysis). Node features include role type and
NLI relation counts broken down by role.

---

**Setup:** place `Fake.csv` and `True.csv` from the ISOT dataset in a `data/` folder.

---

**Refactor note:** all of the pipeline logic described below now lives in the `pipeline/` package next to this notebook, one module per stage. This notebook only calls into that package — see the "Project layout" cell right after this one for what lives where. The narrative markdown cells from the original `models4` notebook are kept in place as documentation for each stage, even though the code itself has moved out.

## Project layout

```
.
├── models4.ipynb          <- this notebook: data loading, dataset construction, model training
├── data/
│   ├── Fake.csv
│   └── True.csv
└── pipeline/
    ├── config.py           # shared constants + pretrained models (ENCODER, NLI_MODEL, spaCy)
    ├── data_loading.py      # load_isot_dataset
    ├── sentence_roles.py    # classify_sentence_roles (claim/evidence/analysis/background)
    ├── retrieval.py         # DuckDuckGo headline search + full-article-text fetching (cached)
    ├── credibility.py       # per-domain source-credibility SQLite DB
    ├── graph_building.py    # build_article_graph (base per-article graph)
    ├── augmentation.py      # augment_with_cross_source, score_documents (search-result augmentation)
    ├── features.py          # compute_node_features, graph_to_pyg
    ├── model.py             # FakeNewsGAT + train_gat/finetune_gat/evaluate_model/save_model/load_model
    ├── dataset_builder.py   # build_dataset, build_batch, retry_failed_evidence
    └── training_loop.py     # batched_train_loop (streaming build+train)
```

**Where to make changes:**
- Want to change how retrieved articles get scored or wired into the graph? → `pipeline/augmentation.py`
- Want to change how the base article graph is built (similarity threshold, NLI usage)? → `pipeline/graph_building.py`
- Want to change the search query, full-article-text fetching, or caching strategy? → `pipeline/retrieval.py`
- Want to change node features or the GAT architecture? → `pipeline/features.py` / `pipeline/model.py`
- Want to change the batching/retry/training-loop strategy? → `pipeline/dataset_builder.py` / `pipeline/training_loop.py`

None of these require touching this notebook — just edit the module and re-run the notebook cells that call it (or restart the kernel if you're using the batched loop, since it holds state across batches).

In [2]:
from sklearn.model_selection import train_test_split
import torch
from torch_geometric.loader import DataLoader

from pipeline.data_loading import load_isot_dataset
from pipeline.dataset_builder import build_dataset, build_batch, retry_failed_evidence
from pipeline.model import FakeNewsGAT, train_gat, finetune_gat, evaluate_model, save_model, load_model
from pipeline.training_loop import batched_train_loop
from pipeline.credibility import bulk_update_from_prediction, print_credibility_leaderboard

print('Pipeline modules loaded.')

Pipeline modules loaded.


## Step 1: Load ISOT Dataset

The ISOT Fake News Dataset contains ~23 000 fake articles (from unreliable sources
flagged by fact-checking organisations) and ~21 000 real articles (from Reuters.com).
We shuffle, binary-encode the label, and strip the Reuters dateline from real articles
to prevent the model from learning a trivial formatting heuristic.

In [3]:
df = load_isot_dataset('data/Fake.csv', 'data/True.csv')
print(f'Total: {len(df)} | Balance: {df["label_binary"].value_counts().to_dict()}')
df[['title', 'text', 'label']].head(2)

Total: 44898 | Balance: {1: 23481, 0: 21417}


,title,text,label
0,Ben Stein Calls Out 9th Circuit Court: Committ...,"21st Century Wire says Ben Stein, reputable pr...",fake
1,Trump drops Steve Bannon from National Securit...,U.S. President Donald Trump removed his chief ...,real


## Step 2: Sentence Role Classification

Each sentence is classified as **claim**, **evidence**, **analysis**, or **background**
using zero-shot NLI. Rather than fine-tuning a dedicated classifier, we run the NLI
model against four defining hypotheses and pick the highest-scoring one.

This is the key structural addition over `models2`. Misinformation rarely fabricates raw
events — it manipulates the *analysis* layer: presenting selective evidence, drawing
unsupported conclusions, or framing neutral facts with loaded interpretation. Making this
distinction explicit in the graph gives the model the right signal to learn from.

**Why not fine-tune a classifier?** Zero-shot NLI generalises across topics without any
labelled role data. A fine-tuned classifier would need a role-labelled news corpus that
doesn't readily exist and would risk overfitting to surface patterns of specific
publications.

*(Implementation: `pipeline/sentence_roles.py`. Run `python -m pipeline.sentence_roles` for a standalone sanity check against a few example sentences.)*

## Step 3: Headline Search

We search using the **article headline** instead of entity queries from the body text.

**Why headlines work better:**
- Headlines are purpose-written to be precise and findable — they are the journalist's best distillation of the article's core claim
- A headline search retrieves articles covering the *same event* from different publishers, which is exactly the cross-source comparison we want
- Entity queries from body text match individual entities that may appear in completely unrelated contexts (hence emojipedia and dndbeyond appearing as evidence)

This reduces from O(n_claims) search calls to O(1) per article.

**Full-text fetching:** search results only come with a one-line snippet, which is thin evidence. For each result, we now also fetch the actual page and extract the article body (via `trafilatura`), so graph construction has many more candidate sentences to draw on instead of just one. Fetches run concurrently and are cached on disk by URL; a page that can't be fetched (paywall, blocked scraper, timeout) just falls back to the snippet rather than failing the whole retrieval.

*(Implementation: `pipeline/retrieval.py`.)*

## Step 4: Source Credibility Database

We maintain a lightweight per-domain **Bayesian credibility score** backed by SQLite.
Each domain starts with a Beta(2, 2) prior (score = 0.5 — no information).

- **Model updates**: when the GAT classifies an article with high confidence, every
  domain that contributed an evidence node receives a fractional update weighted by
  the document's relevance score and the model's confidence.
- **User updates**: explicit human labels can be injected with weight 1.0 (vs 0.3 for
  model updates) to let ground-truth feedback dominate.

The credibility score is blended with the DBSCAN consensus score when ranking retrieved
documents, so high-credibility sources get proportionally more edge weight in the graph.

*(Implementation: `pipeline/credibility.py`.)*

## Step 5: Structured Argumentation Graph

The graph now has two kinds of nodes and three kinds of edges:

**Nodes:**
- `input_*` — sentences from the target article, labeled by role (claim/evidence/analysis/background)
- `ext_*` — sentences from retrieved articles, labeled by role

**Edges:**
- **Intra-article** (input↔input): cosine similarity > threshold, then NLI-typed
- **Cross-source analysis** (input_analysis↔ext_analysis): NLI comparison between target article's analysis and retrieved articles' analysis — this is the core new signal
- **Cross-source evidence** (input_claim↔ext_evidence): NLI comparison between target claims and retrieved evidence

Cross-source edges are only drawn between matching role pairs to avoid noise. Comparing a background sentence to an analysis sentence from another source produces meaningless NLI scores.

Cross-source evidence sentences are drawn from each retrieved article's full text when it could be fetched (falling back to the search snippet otherwise) — see `pipeline/retrieval.py`.

*(Implementation: `pipeline/graph_building.py` for the base graph, `pipeline/augmentation.py` for folding in retrieved evidence.)*

## Step 6: Node Features (20-dimensional)

Each node in the graph is described by a 20-dimensional hand-crafted feature vector.
Features 0–13 were present in `models2`; features 14–19 are new in this version.

| Index | Feature | Description |
|-------|---------|-------------|
| 0 | `n_evidence_nbrs` | Number of external (evidence) neighbours |
| 1 | `n_input_nbrs` | Number of intra-article neighbours |
| 2 | `mean_ev_weight` | Mean document score of evidence neighbours |
| 3 | `max_ev_weight` | Max document score of evidence neighbours |
| 4 | `mean_sim` | Mean edge cosine similarity |
| 5 | `ev_ratio` | Fraction of neighbours that are external |
| 6 | `is_evidence` | 1 if this node is from an external source |
| 7 | `node_weight` | Document credibility / relevance score |
| 8 | `n_entailing` | Count of entailment edges |
| 9 | `n_contradicting` | Count of contradiction edges |
| 10 | `ent_wsum` | Weighted entailment sum (weight × confidence) |
| 11 | `cont_wsum` | Weighted contradiction sum |
| 12 | `ent_ratio` | Entailment fraction of evidence neighbours |
| 13 | `cont_ratio` | Contradiction fraction of evidence neighbours |
| 14–17 | `role_onehot` | One-hot role: claim / evidence / analysis / background |
| 18 | `cross_ent_w` | Cross-source entailment weight (corroboration signal) |
| 19 | `cross_cont_w` | Cross-source contradiction weight (dispute signal) |

*(Implementation: `pipeline/features.py`.)*

## Step 7: Graph Attention Network (GAT)

The classifier is a 4-layer **Graph Attention Network** with the following design choices:

- **Input projection**: a linear layer maps the 20-dim node features into the hidden
  space before any message passing. This decouples feature scale from hidden dimension.
- **4 × GAT2dConv layers** with multi-head attention (4 heads) and skip connections.
  The first three layers use `concat=True` (outputs are concatenated across heads);
  the final layer uses `concat=False` (outputs are averaged) to control the growth of
  the hidden dimension.
- **Jumping Knowledge (JK) aggregation**: representations from all four layers are
  concatenated (`xjk`), so the readout can draw on local *and* long-range structure
  simultaneously.
- **Global pooling**: both `global_mean_pool` and `global_max_pool` are applied to
  `xjk` and concatenated. Mean captures average node behaviour; max captures the most
  extreme signal anywhere in the graph.
- **MLP classifier**: a 3-layer MLP with BatchNorm, ReLU, and Dropout maps the pooled
  graph representation to a single logit (binary cross-entropy loss).
- **Training**: Adam with cosine annealing LR schedule, gradient clipping, and early
  stopping on validation loss.

*(Implementation: `pipeline/model.py`.)*

## Step 8: Build Dataset

For each article we:
1. Tokenise the body into sentences and classify their roles (NLI pass).
2. Search DuckDuckGo with the article headline and retrieve up to 10 results.
3. Score the retrieved documents by DBSCAN consensus + domain credibility.
4. Augment the intra-article graph with cross-source edges (role-matched NLI pairs).
5. Compute the 20-dim node feature matrix and convert to a PyG `Data` object.

> **Note:** this build is slower per article than `models2` because role classification
> adds one full NLI pass per article. However it produces far fewer empty graphs since
> headline search returns topically relevant results for almost every article.

*(Implementation: `pipeline/dataset_builder.py::build_dataset`.)*

In [4]:
pyg_dataset, all_scored_docs = build_dataset(df, sample=500)

Building graphs:   0%|          | 0/500 [00:00<?, ?it/s]

  Fetch failed for "https://en.m.wikipedia.org/wiki/Donald_Trump": fetch_url() got an unexpected keyword argument 'timeout'
  Fetch failed for "https://apnews.com/hub/donald-trump": fetch_url() got an unexpected keyword argument 'timeout'
  Fetch failed for "https://www.whitehouse.gov/": fetch_url() got an unexpected keyword argument 'timeout'
  Fetch failed for "https://apnews.com/article/correspondents-dinner-trump-shoot": fetch_url() got an unexpected keyword argument 'timeout'
  Fetch failed for "https://www.whitehouse.gov/administration/donald-j-trump/": fetch_url() got an unexpected keyword argument 'timeout'
  Fetch failed for "https://www.reuters.com/world/us/donald-trump/": fetch_url() got an unexpected keyword argument 'timeout'
  Fetch failed for "https://time.com/article/2026/07/17/trump-speech-elections-c": fetch_url() got an unexpected keyword argument 'timeout'
  Fetch failed for "https://www.washingtonpost.com/donald-trump/": fetch_url() got an unexpected keyword argume

Building graphs:   1%|          | 5/500 [00:10<14:58,  1.81s/it]

  Fetch failed for "https://www.yelp.com/search?cflt=breakfast_brunch&find_loc=H": fetch_url() got an unexpected keyword argument 'timeout'
  Fetch failed for "https://www.tasteofhome.com/collection/5-ingredient-easy-bre": fetch_url() got an unexpected keyword argument 'timeout'
  Fetch failed for "https://www.yelp.com/search?cflt=breakfast_brunch&find_loc=6": fetch_url() got an unexpected keyword argument 'timeout'
  Fetch failed for "https://www.foodnetwork.com/recipes/photos/our-best-breakfas": fetch_url() got an unexpected keyword argument 'timeout'
  Fetch failed for "https://natashaskitchen.com/best-breakfast-ideas/": fetch_url() got an unexpected keyword argument 'timeout'
  Fetch failed for "https://www.opentable.com/cuisine/best-breakfast-restaurants": fetch_url() got an unexpected keyword argument 'timeout'
  Fetch failed for "https://www.goodhousekeeping.com/food-recipes/g871/quick-bre": fetch_url() got an unexpected keyword argument 'timeout'
  Fetch failed for "https://www

Building graphs:   1%|▏         | 7/500 [00:18<24:44,  3.01s/it]

...

  Fetch failed for "https://www.merriam-webster.com/dictionary/minority": fetch_url() got an unexpected keyword argument 'timeout'
  Fetch failed for "https://en.wikipedia.org/wiki/Minority_group": fetch_url() got an unexpected keyword argument 'timeout'
  Fetch failed for "https://dictionary.cambridge.org/dictionary/english/minority": fetch_url() got an unexpected keyword argument 'timeout'
  Fetch failed for "https://www.britannica.com/topic/minority": fetch_url() got an unexpected keyword argument 'timeout'
  Fetch failed for "https://theworlddata.com/minorities-in-the-us/": fetch_url() got an unexpected keyword argument 'timeout'


Building graphs: 100%|█████████▉| 499/500 [33:57<00:03,  3.45s/it]

  Fetch failed for "https://en.m.wikipedia.org/wiki/Bernie_(2011_film)": fetch_url() got an unexpected keyword argument 'timeout'
  Fetch failed for "https://en.m.wikipedia.org/wiki/Bernie_Sanders": fetch_url() got an unexpected keyword argument 'timeout'
  Fetch failed for "https://m.imdb.com/title/tt1704573/": fetch_url() got an unexpected keyword argument 'timeout'
  Fetch failed for "https://berniesanders.com/": fetch_url() got an unexpected keyword argument 'timeout'
  Fetch failed for "https://thedailyguardian.com/world/us/who-is-bernie-sanders-": fetch_url() got an unexpected keyword argument 'timeout'
  Fetch failed for "https://www.facebook.com/berniesanders/": fetch_url() got an unexpected keyword argument 'timeout'
  Fetch failed for "https://berniesanders.com/about/": fetch_url() got an unexpected keyword argument 'timeout'
  Fetch failed for "https://www.sanders.senate.gov/": fetch_url() got an unexpected keyword argument 'timeout'
  Fetch failed for "https://www.rottentom

Building graphs: 100%|██████████| 500/500 [34:08<00:00,  4.10s/it]


Built 493 | Augmented: 106 | Fallback: 387
Role distribution: {'claim': '75.2%', 'evidence': '0.1%', 'analysis': '9.4%', 'background': '15.3%'}


## Step 8b: Batch Build with Evidence Retry

Two new capabilities replace the monolithic `build_dataset` + single training pass:

### `build_batch`
Builds graphs for one slice of the dataframe. Unlike `build_dataset`, it separates
*failure tracking* from the hot path: articles that return zero evidence are stored in a
`failed_items` list along with their **pre-augmentation graph state** (base graph,
sentence list, role list) so that `retry_failed_evidence` can patch them without
re-running the expensive NLI role-classification pass.

### `retry_failed_evidence`
Runs up to three progressively broader fallback search strategies on failed articles:

1. **Truncated headline** — first six words (removes rare proper nouns that confuse DDG)
2. **NER keyword query** — spaCy extracts named entities + noun chunks → compact query
3. **Lead-sentence query** — first sentence of the article body

Each strategy is tried in order; as soon as one produces documents the graph is
augmented in-place and the item is removed from the failure list. Articles that still
fail after all retries are included as base-graph-only (no cross-source edges) so the
article is not silently dropped from the dataset.


*(Implementation: `pipeline/dataset_builder.py::build_batch` / `retry_failed_evidence`.)*

## Step 8c: Warm-Start Fine-Tuning

`finetune_gat` differs from `train_gat` in three ways:

1. **No re-initialisation** — accepts an existing model and continues from its
   current weights rather than constructing a new `FakeNewsGAT()`.
2. **Lower learning rate** — defaults to `1e-4` (vs `5e-4` for cold start) to
   avoid catastrophic forgetting of patterns learned in earlier batches.
3. **Cosine-annealing restart** — the scheduler's `T_max` is set to the
   fine-tuning epoch count, giving a fresh cosine curve per batch.


*(Implementation: `pipeline/model.py::finetune_gat`.)*

## Step 9b: Batched Training Loop

Replaces the monolithic Step 9 with an iterative loop that interleaves graph
construction and model training:

```
for each batch i:
    1. build_batch()              ← graph construction for this slice
    2. retry_failed_evidence()    ← recover articles with 0 search results
    3. split batch into train/val (for batch-local early stopping)
    4. if i == 0:  train_gat()     (cold start, more epochs)
       else:       finetune_gat()  (warm start, lower LR)
    5. evaluate on fixed global val pool → log per-batch metrics
    6. update credibility DB with this batch's predictions
    7. save checkpoint
```

### Why this helps

- **Faster feedback** — accuracy numbers appear after the first batch, not after
  the entire dataset is built.
- **Compounding DB** — the source credibility database grows richer each batch, so
  later batches score retrieved documents with more accumulated signal.
- **Visualised learning curve** — `batch_history` (accuracy / F1 / AUC per batch on a
  fixed global val pool) makes convergence visible.

### Parameters to tune

| Parameter | Default | Notes |
|-----------|---------|-------|
| `batch_size` | 50 | Articles per batch; smaller = more DB updates, noisier gradients |
| `val_pool_size` | 60 | Fixed global val set; set aside before any batching |
| `cold_epochs` | 80 | Epochs for batch 0 (cold start) |
| `warm_epochs` | 30 | Epochs per subsequent batch (warm start) |
| `val_split` | 0.2 | Fraction of each batch held out for batch-local early stopping |
| `retry` | True | Run `retry_failed_evidence` on each batch |

### Resuming after a kernel restart

```python
model = load_model('fake_news_gat_v4_batch003.pt')
# slice work_df from row batch_size*3 onward and pass to a new batched_train_loop call,
# or loop manually over the remaining batches using finetune_gat().
```


*(Implementation: `pipeline/training_loop.py::batched_train_loop`.)*

In [5]:
# Streaming alternative to the "Step 9" cell below — builds graphs and trains in
# interleaved batches instead of waiting for the whole dataset up front. Not run by
# default in this notebook; uncomment to use it in place of Step 8 + Step 9.
#
# final_model, history = batched_train_loop(
#     df,
#     batch_size    = 50,
#     val_pool_size = 60,
#     cold_epochs   = 80,
#     warm_epochs   = 30,
#     retry         = True,
# )

## Step 9: Train, Evaluate, and Update Credibility DB

We do a 70 / 15 / 15 train/val/test split, train the GAT with early stopping, evaluate
on the held-out test set, and then run inference on the full dataset to update the source
credibility database with the model's predictions.

In [6]:
indices             = list(range(len(pyg_dataset)))
train_idx, temp_idx = train_test_split(indices, test_size=0.3, random_state=42)
val_idx, test_idx   = train_test_split(temp_idx, test_size=0.5, random_state=42)

train_data = [pyg_dataset[i] for i in train_idx]
val_data   = [pyg_dataset[i] for i in val_idx]
test_data  = [pyg_dataset[i] for i in test_idx]
print(f'Split: {len(train_data)} train | {len(val_data)} val | {len(test_data)} test')

model = train_gat(train_data, val_data, epochs=150, patience=20)

metrics, test_probs = evaluate_model(model, test_data)
print(f'\nTest Results:')
print(f'  Accuracy : {metrics["accuracy"]:.3f}')
print(f'  F1       : {metrics["f1"]:.3f}')
print(f'  AUC-ROC  : {metrics["auc"]:.3f}')

# Update credibility DB
model.eval()
n_db = 0
with torch.no_grad():
    for batch, scored in zip(DataLoader(pyg_dataset, batch_size=1), all_scored_docs):
        if not scored:
            continue
        prob = torch.sigmoid(model(batch.x, batch.edge_index, batch.batch).squeeze()).item()
        conf = abs(prob - 0.5) * 2
        n_db += bulk_update_from_prediction(scored, prob, conf, confidence_threshold=0.1)

print(f'\nCredibility DB: {n_db} entries updated.')
save_model(model, 'fake_news_gat_v3.pt')

Split: 345 train | 74 val | 74 test
Epoch  10  train=0.5354  val=0.6266  lr=4.95e-04
Epoch  20  train=0.5303  val=0.6372  lr=4.78e-04
Epoch  30  train=0.5333  val=0.6348  lr=4.52e-04
Early stopping at epoch 35

Test Results:
  Accuracy : 0.716
  F1       : 0.687
  AUC-ROC  : 0.847

Credibility DB: 741 entries updated.
Saved to fake_news_gat_v3.pt


In [7]:
print_credibility_leaderboard()

Domain                                    Score  Signals   Model   User
----------------------------------------------------------------------
en.wikipedia.org                          0.603       18     327      0
en.m.wikipedia.org                        0.527       15     303      0
britannica.com                            0.488        9     199      0
whitehouse.gov                            0.526       10     184      0
apnews.com                                0.537       10     178      0
youtube.com                               0.516        5     145      0
merriam-webster.com                       0.424        5     129      0
usatoday.com                              0.504        5     103      0
dictionary.cambridge.org                  0.542        5      96      0
time.com                                  0.507        4      86      0
worldatlas.com                            0.538        3      76      0
amazon.com                                0.408        3      61 

## Architecture Improvement Proposals

These are concrete changes that could improve the model, in rough order of
expected impact.

---

### 1. Replace `GATConv` with `GATv2Conv` *(high impact, trivial change)*

The original GAT attention mechanism has a theoretical limitation: it computes attention
weights before combining the query and key representations, making it equivalent to a
static (input-independent) attention in certain graph structures. GATv2 fixes this by
applying the non-linearity *after* concatenating the node representations, making
attention genuinely dynamic.

```python
# Change in imports:
from torch_geometric.nn import GATv2Conv  # replaces GATConv

# Change in __init__: replace GATConv(...) → GATv2Conv(...)
# The API is identical; no other changes needed.
self.conv1 = GATv2Conv(hidden_dim, hidden_dim, heads=n_heads, concat=True, dropout=dropout)
```

---

### 2. Feed edge features into attention *(medium impact)*

The graph carries rich edge attributes (edge type, NLI confidence, similarity, cross-source flag)
but `GATConv` / `GATv2Conv` can only use them if you pass `edge_dim`. Currently they are
computed but ignored during message passing.

```python
EDGE_DIM = 7  # matches the edge_attr dimension in graph_to_pyg()

self.conv1 = GATv2Conv(hidden_dim, hidden_dim, heads=n_heads,
                       concat=True, dropout=dropout, edge_dim=EDGE_DIM)
# ... same for conv2, conv3, conv4

# In forward():
x1 = self.bn1(F.relu(self.lin1(self.conv1(x, edge_index, edge_attr=edge_attr)))) + x
```

You would also need to add `edge_attr` as a parameter to `forward()` and pass `b.edge_attr`
in the training loop.

---

### 3. Use a richer sentence encoder *(medium impact)*

`all-MiniLM-L6-v2` is fast but relatively weak for semantic nuance. Upgrading to
`all-mpnet-base-v2` (same API, 420 MB vs 80 MB) consistently gives +2–4 points on
STS benchmarks and would produce better embeddings for both the graph edges and
NLI role classification.

```python
ENCODER = SentenceTransformer('all-mpnet-base-v2')
```

Alternatively, `BAAI/bge-small-en-v1.5` is only marginally larger than MiniLM but
significantly stronger.

---

### 4. Replace column-max normalisation with Z-score standardisation *(low-medium impact)*

The current normalisation divides each feature column by its maximum value. This is
sensitive to outliers (one very large value collapses all others toward 0) and doesn't
centre the features, which can slow down learning.

```python
from sklearn.preprocessing import StandardScaler

# In build_dataset, after collecting all pyg_data:
all_x = torch.cat([d.x for d in pyg_data], dim=0).numpy()
scaler = StandardScaler().fit(all_x)

for d in pyg_data:
    d.x = torch.tensor(scaler.transform(d.x.numpy()), dtype=torch.float)
```

Save the scaler alongside the model weights so you can normalise at inference time.

---

### 5. Add a heterogeneous graph formulation *(high impact, more work)*

Right now, `input` and `evidence` nodes are structurally identical in the model — only
feature 6 (`is_evidence`) distinguishes them. PyG's `HeteroData` lets you define
separate embedding spaces and message-passing weights for different node/edge types,
which is a much more principled way to handle this.

```python
from torch_geometric.data import HeteroData
from torch_geometric.nn import HeteroConv, SAGEConv

data = HeteroData()
data['input'].x    = input_features
data['evidence'].x = evidence_features
data['input',  'similar_to', 'input'].edge_index    = intra_edges
data['input',  'supported_by', 'evidence'].edge_index = cross_edges
data['input',  'contradicted_by', 'evidence'].edge_index = contra_edges
```

This is the most architecturally significant change but also the largest refactor.

---

### 6. Add a role-prediction auxiliary loss *(low-medium impact)*

If you have any ground-truth role labels (or can create a small labelled sample with
`classify_sentence_roles` as a noisy teacher), you can add a node-level auxiliary loss
that forces the model to learn role-aware representations. Multi-task learning often
improves the primary task even when the auxiliary task is noisy.

```python
# In forward(), add a branch from the node embeddings before pooling:
role_logits = self.role_head(xjk)  # shape (n_nodes, 4)

# Training loss:
loss = bce_loss(graph_logit, label) + 0.1 * ce_loss(role_logits, node_role_labels)
```